## Import Libraries

In [1]:
import os
import io 
import sys 
from dotenv import load_dotenv
from openai import OpenAI 
import gradio as gr 
import subprocess
from IPython.display import Markdown, display

## Load API keys from the environment

In [2]:
load_dotenv(override = True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-


## Set up the OpenAI and Anthropic clients

In [3]:
openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)

## Define the available models

In [4]:
models = ["gpt-5", "claude-sonnet-4-5-20250929"]
clients = {"gpt-5": openai, "claude-sonnet-4-5-20250929": anthropic}

## Gather system information about this machine

In [5]:
from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '25.5.0',
  'version': 'Darwin Kernel Version 25.5.0: Tue Jun  9 22:28:17 PDT 2026; root:xnu-12377.121.10~1/RELEASE_ARM64_T8142',
  'kernel': '25.5.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin25.5.0'},
 'package_managers': ['xcode-select (CLT)', 'brew'],
 'cpu': {'brand': 'Apple M5',
  'cores_logical': 10,
  'cores_physical': 10,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'g++': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'clang': 'Apple clang version 21.0.0 (clang-2100.1.1.101)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

## Ask the model how to compile and run C++ on this system

In [7]:
message = f""" 
Here is a report of the system information for my computer.
I want to run a C++ compiler to compile a single C++ file called main.cpp and then execute it in the simplest way possible.
Please reply with whether I need to install any C++ compiler to this. if so, please provide the simplest step by step instructions to do so.

if I'm already set up to compile C++ code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.

System information:
{system_info}
"""

response = openai.chat.completions.create(model = models[0], messages= [{"role": "user", "content": message }])
display(Markdown(response.choices[0].message.content))

You’re already set up. Your system has Apple Clang (C/C++ compiler) installed via the Xcode Command Line Tools:
- clang/gcc/g++: Apple clang version 21.0.0

No installation is needed to compile and run a single C++ file.

For fastest typical runtime without sacrificing correctness, use optimized build with ThinLTO:

- compile_command (Python list for subprocess.run):
  ["clang++", "-std=c++20", "-O3", "-flto=thin", "-DNDEBUG", "main.cpp", "-o", "main"]

- run_command:
  ["./main"]

Notes:
- Using clang++ is preferred on macOS; g++ on your system is just a Clang front-end.
- If you ever hit a linker complaining about -flto=thin (unlikely on your setup), drop that flag or replace with -flto.

## Define the compile and run commands

In [6]:
compile_command = ["clang++", "-std=c++20", "-O3", "-DNDEBUG", "-flto=thin", "main.cpp", "-o", "main"]
run_command = ["./main"]

## Define the prompts used to port Python to C++

In [7]:
language = "C++"
extension = "cpp"

system_prompt = f"""
Your task is to convert Pyhton code into high performance {language} code. 
Respond only with {language} code. Do not provide any explanation other than occasional comments. 
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f""" 
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time. 
The system information is: 
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code. 
Python code to port:

```python
{python}
```
"""

## Build the full message list for the API call

In [8]:
def message_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]

## Save the generated C++ code to a file

In [9]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

## Call the model to port the Python code to C++

In [10]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages = message_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    return reply

## Run the Python code and capture its output

In [11]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout 
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

## Compile and run the generated C++ code

In [12]:
def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check = True, text = True, capture_output = True)
        run_result = subprocess.run(run_command, check = True, text = True, capture_output= True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

## Example Python code, a slow pi calculation

In [13]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

## Build the Gradio interface

In [14]:
from styles import CSS 

with gr.Blocks(css = CSS, theme = gr.themes.Monochrome(), title = f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale = 6):
            python = gr.Code(
                label = "Python (original)",
                value = pi,
                language="python",
                lines = 26
            )
        with gr.Column(scale = 6):
            cpp = gr.Code(
                label = f"{language} (generated)",
                value = "",
                language = "cpp",
                lines = 26
            )
    
    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value = models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height = True):
        with gr.Column(scale = 6):
            python_out = gr.TextArea(label = "Python result", lines = 8, elem_classes= ["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label = f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn = port, inputs = [model, python], outputs = [cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

            

## Launch the app

In [15]:
ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
